# 03 — LSTM next-day forecasting (optional TensorFlow)
**New 2026 implementation; not the model code used for the published paper.**

This notebook uses a fixed lookback of 30 observed sessions. Scalers are learned from the training period **only**. Validation/test sequences may reference earlier **feature values** as historical context, but target dates are assigned strictly to their own time periods. No hidden state or training sample may contain later target information. New results are not the historical MAE 0.1798 / RMSE 0.1873.

In [ ]:
from pathlib import Path
from sp500_research import load_prices, download_sp500, build_supervised_frame, chronological_split
DATA_FILE = Path('../data/sp500_ohlcv.csv')
if DATA_FILE.exists():
    prices = load_prices(DATA_FILE)
else:
    # Explicit opt-in: new Yahoo Finance download, not the original 2024 data vintage.
    prices = download_sp500(start='2000-01-01', end='2025-01-01')
print('Data coverage:', prices.index.min().date(), 'to', prices.index.max().date(), len(prices), 'daily rows')

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sp500_research.features import model_features
from sp500_research.evaluation import regression_metrics
frame=build_supervised_frame(prices)
parts=chronological_split(frame)
cols=model_features(frame)
scaler_x = StandardScaler().fit(parts.train[cols])
scaler_y = StandardScaler().fit(parts.train[['target_next_close']])
X = scaler_x.transform(frame[cols]).astype('float32')
y = scaler_y.transform(frame[['target_next_close']]).astype('float32').reshape(-1)
dates = frame.target_date.to_numpy()
LOOKBACK=30

def sequence_arrays(X,y,dates,lookback):
    xseq, yseq, target_dates=[],[],[]
    for end in range(lookback-1,len(X)):
        xseq.append(X[end-lookback+1:end+1])
        yseq.append(y[end])
        target_dates.append(dates[end])
    return np.stack(xseq),np.array(yseq),np.array(target_dates)

Xseq,yseq,sequence_dates=sequence_arrays(X,y,dates,LOOKBACK)
cut_train=parts.train.target_date.max().to_datetime64()
cut_valid=parts.valid.target_date.max().to_datetime64()
train_mask=sequence_dates<=cut_train
valid_mask=(sequence_dates>cut_train)&(sequence_dates<=cut_valid)
test_mask=sequence_dates>cut_valid
print('Sequence samples:',train_mask.sum(),valid_mask.sum(),test_mask.sum())
assert sequence_dates[train_mask].max()<sequence_dates[valid_mask].min()
assert sequence_dates[valid_mask].max()<sequence_dates[test_mask].min()

In [ ]:
try:
    import tensorflow as tf
except ImportError as exc:
    raise ImportError('This optional notebook requires tensorflow. Install it separately to run LSTM training.') from exc

tf.random.set_seed(42)
model=tf.keras.Sequential([
    tf.keras.layers.Input(shape=(LOOKBACK,len(cols))),
    tf.keras.layers.LSTM(24),
    tf.keras.layers.Dense(1)
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=.001),loss='mse')
history=model.fit(Xseq[train_mask],yseq[train_mask],
                  validation_data=(Xseq[valid_mask],yseq[valid_mask]),
                  epochs=15,batch_size=32,shuffle=False,verbose=0,
                  callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=3,
                                                             restore_best_weights=True)])
print('Training epochs:',len(history.history['loss']))

In [ ]:
actual=scaler_y.inverse_transform(yseq[test_mask].reshape(-1,1)).ravel()
pred=scaler_y.inverse_transform(model.predict(Xseq[test_mask],verbose=0)).ravel()
print('New test results (index points):',regression_metrics(actual,pred))
# The current-day close is the naive t->t+1 price-level baseline.
naive_close=frame.loc[frame.target_date.isin(sequence_dates[test_mask]),'close_t'].to_numpy()
print('Naive last-close results:',regression_metrics(actual,naive_close))

### Limits
These training settings are illustrative and not optimized. Compare with the original publication only after matching its data vintage, target definition, forecast horizon, features and loss scaling. Macro data are deliberately omitted because release-time availability has not been verified.